# Retail Analytics — Customer Segmentation & Sales Optimization

**Author:** Sagar Kandelkar  
**Date:** September 2026  
**Dataset:** Synthetic Retail Data (20 customers, 25 transactions)

## Objectives
1. Perform RFM (Recency, Frequency, Monetary) customer segmentation
2. Analyze sales trends and product performance
3. Evaluate campaign effectiveness
4. Identify churn risk customers
5. Generate actionable insights for marketing and operations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported successfully')

In [ ]:
# Load datasets
customers = pd.read_csv('../data/customers.csv')
transactions = pd.read_csv('../data/transactions.csv')
products = pd.read_csv('../data/products.csv')
campaigns = pd.read_csv('../data/campaigns.csv')

print('Customers:', customers.shape)
print('Transactions:', transactions.shape)
print('Products:', products.shape)
print('Campaigns:', campaigns.shape)

## 1. Customer Segmentation Analysis (RFM)

In [ ]:
# Segment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segment_counts = customers['customer_segment'].value_counts()
colors = ['#1a5fb4', '#26a269', '#e5a50a', '#c01c28', '#9c5c5c', '#7c3aed', '#f97316']
axes[0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.0f%%', 
          colors=colors[:len(segment_counts)], startangle=90)
axes[0].set_title('Customer Segment Distribution', fontsize=14, fontweight='bold')

churn_counts = customers['churn_risk_score'].value_counts()
axes[1].bar(churn_counts.index, churn_counts.values, color=['#26a269', '#e5a50a', '#c01c28'])
axes[1].set_title('Churn Risk Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# RFM Score analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(customers['recency_days'], bins=8, color='#1a5fb4', edgecolor='white')
axes[0].set_title('Recency (Days Since Last Purchase)')
axes[0].set_xlabel('Days')

axes[1].hist(customers['frequency'], bins=8, color='#26a269', edgecolor='white')
axes[1].set_title('Frequency (Total Transactions)')
axes[1].set_xlabel('Transactions')

axes[2].hist(customers['monetary_total']/1000, bins=8, color='#e5a50a', edgecolor='white')
axes[2].set_title('Monetary (Total Spend in ₹K)')
axes[2].set_xlabel('Amount (₹000s)')

plt.tight_layout()
plt.show()

print('\nRFM Summary:')
print(customers[['recency_days', 'frequency', 'monetary_total']].describe().round(2))

## 2. Sales Performance Analysis

In [ ]:
# Category-wise sales
category_sales = transactions.groupby('product_category').agg({
    'total_amount': 'sum',
    'transaction_id': 'count'
}).rename(columns={'transaction_id': 'transactions'}).sort_values('total_amount', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(category_sales.index, category_sales['total_amount'], color=['#1a5fb4', '#26a269', '#e5a50a', '#c01c28'])
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].set_ylabel('Revenue (₹)')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(category_sales.index, category_sales['transactions'], color=['#1a5fb4', '#26a269', '#e5a50a', '#c01c28'])
axes[1].set_title('Transactions by Category', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print(category_sales)

In [ ]:
# Channel & Payment Analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

channel_sales = transactions.groupby('channel')['total_amount'].sum()
axes[0].pie(channel_sales.values, labels=channel_sales.index, autopct='%1.1f%%', 
          colors=['#1a5fb4', '#26a269'], startangle=90)
axes[0].set_title('Revenue by Channel', fontweight='bold')

payment_counts = transactions['payment_method'].value_counts()
axes[1].barh(payment_counts.index, payment_counts.values, color='#3b82f6')
axes[1].set_title('Payment Method Distribution', fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. Campaign Effectiveness

In [ ]:
# Campaign ROI
campaigns['roi'] = ((campaigns['actual_revenue'] - campaigns['budget_inr']) / campaigns['budget_inr'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_campaign = ['#1a5fb4', '#26a269', '#e5a50a', '#c01c28', '#9c5c5c', '#7c3aed']
bars = axes[0].bar(campaigns['campaign_name'], campaigns['roi'], color=colors_campaign)
axes[0].set_title('Campaign ROI (%)', fontweight='bold')
axes[0].set_ylabel('ROI %')
axes[0].tick_params(axis='x', rotation=30, ha='right')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].bar(campaigns['campaign_name'], campaigns['actual_revenue']/100000, color=colors_campaign)
axes[1].set_title('Actual Revenue (₹ Lakhs)', fontweight='bold')
axes[1].set_ylabel('Revenue (₹L)')
axes[1].tick_params(axis='x', rotation=30, ha='right')

plt.tight_layout()
plt.show()

print('\nCampaign Performance Summary:')
print(campaigns[['campaign_name', 'budget_inr', 'actual_revenue', 'roi']].to_string(index=False))

## 4. Product Performance

In [ ]:
# Top products by revenue
product_perf = transactions.groupby('product_name').agg({
    'total_amount': 'sum',
    'quantity': 'sum'
}).sort_values('total_amount', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(product_perf)), product_perf['total_amount'], color='#1a5fb4')
ax.set_yticks(range(len(product_perf)))
ax.set_yticklabels(product_perf.index[::-1])
ax.set_xlabel('Revenue (₹)')
ax.set_title('Top 10 Products by Revenue', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Insights & Recommendations

In [ ]:
print('='*60)
print('RETAIL ANALYTICS — KEY INSIGHTS')
print('='*60)

print(f"\n1. CUSTOMER SEGMENTS:")
for seg, count in customers['customer_segment'].value_counts().items():
    revenue = customers[customers['customer_segment']==seg]['monetary_total'].sum()
    print(f"   {seg}: {count} customers (₹{revenue/1000:.0f}K revenue)")

print(f"\n2. CHURN RISK:")
high_risk = customers[customers['churn_risk_score']=='High']
print(f"   High Risk Customers: {len(high_risk)} ({len(high_risk)/len(customers)*100:.0f}%)")
print(f"   At-Risk Revenue: ₹{high_risk['monetary_total'].sum()/1000:.0f}K")

print(f"\n3. SALES CHANNELS:")
online_pct = transactions[transactions['channel']=='Online']['total_amount'].sum() / transactions['total_amount'].sum() * 100
print(f"   Online: {online_pct:.1f}% | Store: {100-online_pct:.1f}%")

print(f"\n4. CAMPAIGN ROI:")
avg_roi = campaigns['roi'].mean()
print(f"   Average Campaign ROI: {avg_roi:.0f}%")
best_campaign = campaigns.loc[campaigns['roi'].idxmax(), 'campaign_name']
print(f"   Best Performing: {best_campaign}")

print(f"\n5. TOP PRODUCT CATEGORY:")
top_cat = category_sales.index[0]
print(f"   {top_cat}: ₹{category_sales.loc[top_cat, 'total_amount']/1000:.0f}K revenue")

print('\n' + '='*60)
print('RECOMMENDATIONS')
print('='*60)
print("\n1. Focus retention efforts on 'At Risk' and 'Cannot Lose Them' segments")
print("2. Increase online channel investment (higher revenue share)")
print("3. Replicate FLASH24 strategy (highest ROI) for future campaigns")
print("4. Cross-sell Electronics to Apparel customers (complementary categories)")
print("5. Implement automated win-back for 'Hibernating' customers")
print('='*60)

---

**Conclusion:** This analysis demonstrates how customer segmentation, sales analytics, and campaign tracking can drive retail growth. The RFM framework provides a clear lens for prioritizing marketing investment and retention efforts.

**Next Steps:**
- Deploy automated RFM scoring pipeline
- Launch targeted campaigns per segment
- Build real-time dashboard for ongoing monitoring
- A/B test personalized offers for churn-risk customers